In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy
import scipy

In [ ]:

NUM_SEEDS = 20
DIMS = [2, 5, 10, 20 ,30]
DIMS = [5, 10, 20 ,30,50]

LESLOGEIONLY = False

WITHIN_MDL = False

function = 'gpsample_gpytorchMDL'#,'square' #'gpsample_gpytorchMDL'
#function = 'gpsample_within_mdl'
function = 'gpsample_within_mdl_20250512'


avrg_best_histories = []
lower_quantiles_histories = []
upper_quantiles_histories = []
avrg_rank_histories = []
significantly_worse_at_end_histories = []
cum_avrg_best_histories = []
cum_lower_quantiles_histories = []
cum_upper_quantiles_histories = []
cum_significantly_worse_at_end_histories = []
cum_avrg_rank_histories = []


 
function_names = [r'\makecell{\textbf{high}: $\mathrm{p}(l) =$ \\   $\mathrm{logn}(-2.5\sqrt{2} + \mathrm{log} \sqrt{d},\sqrt{3}/5)$ }', # \\ \mathbb{E}[\mathrm{p}(l)] =  
                  r'\makecell{\textbf{medium}: $\mathrm{p}(l) =$ \\   $\mathrm{logn}(-2.0\sqrt{2} + \mathrm{log} \sqrt{d},\sqrt{3}/4)$ }',
                  r'\makecell{\textbf{low}: $\mathrm{p}(l) =$ \\   $\mathrm{logn}(-1.0\sqrt{2} + \mathrm{log} \sqrt{d},\sqrt{3}/2)$ }',
                 r'\makecell{\textbf{extremely low} \cite{hvarfner2024vanilla}: $\mathrm{p}(l) =$ \\   $\mathrm{logn}(1.0\sqrt{2} + \mathrm{log} \sqrt{d},\sqrt{3})$ }']


excpected_ls = np.array([[0.08, 0.11, 0.15, 0.19, 0.25],[0.16, 0.23,0.33,0.40,0.52],[0.83, 1.19, 1.67, 2.05, 2.65],[21.86,30.92,34.73,53.56,69.15]]) # medium, low, ext_low
# wilson within model length scales: 0.56, 0.79, 1.18, 1.37,  1.77   
if WITHIN_MDL:
    functions = ['within_mdl_high/gpsample','within_mdl_medium/gpsample','within_mdl_low/gpsample','within_mdl_ext_low/gpsample']
else:
    #functions = ['oom_HVARFNER_HYPERPRIOR_very2_complex/gpsample','oom_HVARFNER_HYPERPRIOR_very_complex/gpsample','oom_HVARFNER_HYPERPRIOR_complex/gpsample','gpsample_oom_HVARFNER_HYPERPRIOR/gpsample']
    functions = ['oom_high/gpsample','oom_medium/gpsample','oom_low/gpsample','oom_ext_low/gpsample']

for CUM_RES in [True, False]:
    for function in functions:
        DISTR_PATH = "./Data/"+function+"/local_optima_distribution/"
        BEST_HIST_PATH = "./Data/"+function+"/optimizer_history/list_of_bests_"
        SAMPLED_POINTS_PATH = "./Data/"+function+"/sampled_data/sampled_data_history_"
        LENGTH_SCALE_PATH = "./Data/"+function+"/length_scale/length_scale_"






        if LESLOGEIONLY:
            METHODS = ['les_250_8','logei','turbo', 'sobol']  
        else:
            if WITHIN_MDL:
                METHODS = ['les_250_8','mes','logei','turbo','hci_gibo_09', 'sobol'] # final selection of algorithms #,'hci_gibo_09'
            else:
                METHODS = ['les_250_8','mes','logei','loghvarei','turbo','hci_gibo_09', 'sobol']





        LABEL_NAMES = {'tracing':'With gradient tracing',
        'mes':'MES',
        'logei':'logEI',
        'loghvarei':'logEI-DSP',
        'turbo':'TuRBO',
        'sobol':'Sobol random',
        'std_gibo':'GIBO',
        'hci_gibo':'HCI-GIBO',
        'hci_gibo_09':'HCI-GIBO',
        'les_20_8':'LES-ADAM: L = 20, P = 8',
        'les_250_4':'LES: L = 250, P = 4',
        'les_250_16':'LES: L = 250, P = 16',
        'les_250_8':'LES (ours)',
        'les_fp_wgrad':'LES-ADAM-Opt.Cond.: L = 250, P = 8 ',
        'lesGD_250_8':'LES-GD: L = 250, P = 8',
        'lesgradcond_20_8':'LES-ADAM-Grad.-Cond.: L = 20, P = 8',
        'localTS':'Local Thompson Sampling',
        'lesCMAES_20_8':'LES-CMAES: L = 20, P = 8'}

        METHOD_TABLE_NAMES = []
        for method in METHODS:
            METHOD_TABLE_NAMES.append(LABEL_NAMES[method])


        def decompress_gibo(df):
            data = df[['y','n']].to_numpy(dtype=float)
            repeats = np.diff(data[:,-1].astype(int))
            repeats = np.insert(repeats, 0, data[0,-1])
            repeated = np.repeat(data[:, :-1], repeats, axis=0)
            return np.minimum.accumulate(repeated)

        ######################## calculate median and quantiles of the best observed function value ##############

        from matplotlib.lines import Line2D

        avrg_best_history = []
        std_best_history = []
        lower_quantiles_history =[]
        upper_quantiles_history =[]
        avrg_rank_history = []
        significantly_worse_at_end_history = []
        avrg_time_deltas = []
        std_time_deltas = []

    

        for dim in DIMS: 
            data_mean = []
            stds = []

            time_deltas_mean_per_method = []
            time_deltas_std_per_method = []
            lower_quantile =[]
            upper_quantile =[]


            num_objective_calls = min(20*dim, 400)

            all_y_data_per_dim = np.zeros((NUM_SEEDS,len(METHODS),num_objective_calls))


            for method in range(len(METHODS)):
                y_data = np.zeros((0,0))
                # Collect timestamp differences for all seeds having a timestamp column
                time_arrays = []
                for seed in range(NUM_SEEDS):
                    

                    
                    file_identifier = f'{(seed+1):05d}_{dim}_{METHODS[method]}.csv'
                    if CUM_RES:
                        try: 
                            table = pd.read_csv(SAMPLED_POINTS_PATH + file_identifier) 
                        except: 
                            print(f'Unable to find file {SAMPLED_POINTS_PATH+file_identifier}.')
                            #continue
                    else:
                        try: 
                            table = pd.read_csv(BEST_HIST_PATH + file_identifier) 
                        except: 
                            print(f'Unable to find file {BEST_HIST_PATH+file_identifier}.')
                            #continue
                        
                    if METHODS[method] == 'std_gibo' or METHODS[method] == 'hci_gibo' or METHODS[method] == 'hci_gibo_09' and not CUM_RES :
                        new_data = decompress_gibo(table.dropna())
                    else:
                        if CUM_RES:
                            new_data = np.reshape(table['y'].to_numpy(), [-1,1])
                            if np.isnan(new_data[-1]):
                                new_data[-1] = new_data[-2]
                        else:
                            new_data = np.reshape(table['f'].to_numpy(), [-1,1])
                    

                    if CUM_RES:
                        if y_data.shape[0] == 0:
                            y_data = np.expand_dims(np.cumsum(new_data[:num_objective_calls, :]),axis = 1)
                        else:
                            try: 
                                new_data = np.expand_dims(np.cumsum(new_data[:num_objective_calls, :]),axis = 1)
                                y_data = np.concatenate([y_data, new_data], axis=1)
                            except:
                                y_data = np.concatenate([y_data, np.expand_dims(y_data[:,-1],1)], axis=1)
                                print(METHODS[method])
                                print('data missing augmenting with exisiting data set')

                    else:
                        if y_data.shape[0] == 0:
                            y_data = new_data[:num_objective_calls, :]
                        else: 
                            new_data = new_data[:num_objective_calls, :]
                            y_data = np.concatenate([y_data, new_data], axis=1)

                    if 'timestamp' in table.columns:
                        # Crop timestamps to match the # of objective calls
                        timestamps = table['timestamp'].to_numpy()[:num_objective_calls]
                        # Compute the consecutive differences
                        if len(timestamps) > 1:
                            diffs = np.diff(timestamps)
                            time_arrays.append(diffs)


                    
                data_mean.append(np.median(y_data, axis=1))
                stds.append(np.std(y_data, axis=1))
                all_y_data_per_dim[:,method,0:y_data.shape[0]] = np.transpose(y_data)
                for i_add in range(y_data.shape[0],num_objective_calls,1):
                    all_y_data_per_dim[:,method,i_add] = all_y_data_per_dim[:,method,y_data.shape[0]-1] 
                try:
                    lower_quantile.append(np.quantile(y_data, 0.25, axis=1))
                    upper_quantile.append(np.quantile(y_data, 0.75, axis=1))
                except:
                    lower_quantile.append([])
                    upper_quantile.append([])
                    
                stds.append(np.std(y_data, axis=1))

                # Collect mean/std across seeds for the time-deltas (if available)
                if len(time_arrays) > 0:
                    time_arrays = np.array(time_arrays)  # shape: [num_seeds_with_timestamps, num_objective_calls-1]
                    time_deltas_mean_per_method.append(np.mean(time_arrays, axis=0))
                    time_deltas_std_per_method.append(np.std(time_arrays, axis=0))
                else:
                    # No timestamp data for this method in this dimension
                    time_deltas_mean_per_method.append(None)
                    time_deltas_std_per_method.append(None)


            avrg_best_history.append(data_mean)
            #avrg_best_history_normalized.append(data_mean_normalized)
            std_best_history.append(stds)
            #std_best_history_normalized.append(stds_normalized)
            lower_quantiles_history.append(lower_quantile)
            upper_quantiles_history.append(upper_quantile)

            avrg_time_deltas.append(time_deltas_mean_per_method)
            std_time_deltas.append(time_deltas_std_per_method)

            # calculate ranks
            tmp_sorted = np.argsort(all_y_data_per_dim,axis = 1)
            ranks_per_dim = np.zeros(tmp_sorted.shape)
            


            for i_method in range(len(METHODS)):
                res0,res1,res2 = np.where(tmp_sorted == i_method)
                ranks_per_dim[res0,i_method,res2] = res1

            ranks_per_dim += 1
            # calculate average rank across seeds
            avrg_rank_history.append(np.mean(ranks_per_dim,axis=0)) 
            lowest_average_rank_ind_per_dim = np.argmin(np.mean(ranks_per_dim,axis=0),axis=0)
            
    

            


            significantly_worse_at_end = np.zeros(len(METHODS))

            for method in range(len(METHODS)): # check wether the respective algorithms are statistically significantly worse then the best one
                if method == lowest_average_rank_ind_per_dim[-1]:
                    significantly_worse_at_end[method] = 0
                else:
                    best_results = all_y_data_per_dim[:,lowest_average_rank_ind_per_dim[-1],-1]
                    method_results = all_y_data_per_dim[:,method,-1] 
                    test_result = scipy.stats.wilcoxon(best_results, method_results, alternative='less', axis=0, nan_policy='propagate', keepdims=False)
                    significantly_worse_at_end[method] = test_result.pvalue < 0.05 #95 % significance level
            significantly_worse_at_end_history.append(copy.deepcopy(significantly_worse_at_end)) 
            

        if CUM_RES:
            cum_avrg_best_histories.append(copy.deepcopy(avrg_best_history))
            cum_lower_quantiles_histories.append(copy.deepcopy(lower_quantiles_history))
            cum_upper_quantiles_histories.append(copy.deepcopy(upper_quantiles_history))
            cum_avrg_rank_histories.append(copy.deepcopy(avrg_rank_history))
            cum_significantly_worse_at_end_histories.append(copy.deepcopy(significantly_worse_at_end_history))
        else:
            avrg_best_histories.append(copy.deepcopy(avrg_best_history))
            lower_quantiles_histories.append(copy.deepcopy(lower_quantiles_history))
            upper_quantiles_histories.append(copy.deepcopy(upper_quantiles_history))   
            avrg_rank_histories.append(copy.deepcopy(avrg_rank_history))

            significantly_worse_at_end_histories.append(copy.deepcopy(significantly_worse_at_end_history))


    


    






In [ ]:
def get_last_non_nan_value(data_array):
    """
    Returns the last non-NaN entry of a 1D NumPy array.
    If all entries are NaN, returns None.
    """
    if data_array is None or len(data_array) == 0:
        return None
    not_nan_indices = np.where(~np.isnan(data_array))[0]
    if len(not_nan_indices) == 0:
        return None
    return data_array[not_nan_indices[-1]]

def gather_all_medians(median_results):
    """
    Go through the entire median_results structure and collect all
    last non-NaN values in one list (ignoring None).
    This helps us find the min/max for color mapping.
    """
    all_medians = []
    for func_idx in range(len(median_results)):
        for dim_idx in range(len(median_results[func_idx])):
            for method_idx in range(len(median_results[func_idx][dim_idx])):
                val = get_last_non_nan_value(median_results[func_idx][dim_idx][method_idx])
                if val is not None:
                    all_medians.append(val)
    return all_medians

def interpolate_color(value, vmin, vmax):
    """
    Linearly maps 'value' in [vmin, vmax] to a color ranging from (0,0,1) [blue]
    to (1,0.65,0) [orange]. If vmin == vmax, returns blue (0,0,1).

    Return tuple (R, G, B) in [0, 1].
    """
    if vmin == vmax:
        # If there's only one median value in the entire dataset,
        # just return blue.
        return (0.0, 0.0, 1.0)

    # Normalize to [0,1]
    ratio = (value - vmin) / (vmax - vmin)
    # (R,G,B) = (0,0,1) -> (1,0.65,0)
    R = 0.0 + ratio * (1.0 - 0.0)       # 0 -> 1
    G = 0.0 + ratio * (0.65 - 0.0)      # 0 -> 0.65
    B = 1.0 - ratio * (1.0 - 0.0)       # 1 -> 0
    return (R, G, B)

def color_text_with_median(median_val, text, overall_min, overall_max):
    """
    Wrap the given 'text' in a latex color command based on the median_val,
    mapping from overall_min (blue) to overall_max (orange).

    If median_val is None, returns "-" without color.
    """
    if median_val is None:
        return "-"

    (r, g, b) = interpolate_color(median_val, overall_min, overall_max)
    # Format color with 2 decimals in the rgb specification
    return rf"\textcolor[rgb]{{{r:.2f},{g:.2f},{b:.2f}}}{{{text}}}"

###############################################################################
# Main table creation function
###############################################################################

def create_latex_table(function_names, DIMS, METHODS,
                       median_results_list,
                       lower_quantiles_list,
                       upper_quantiles_list,
                       data_type_flag,
                       use_multirow=False,
                       significantly_worse_at_end = None,
                       expected_ls = None):


    # 1) Find global min/max among all medians for color mapping
    all_medians = gather_all_medians(median_results_list)
    if len(all_medians) == 0:
        # If no medians at all, handle gracefully
        global_min = 0
        global_max = 1
    else:
        global_min = min(all_medians)
        global_max = max(all_medians)

    # find best performing algorithm for each dimension and complexity:

    # Start building the table
    latex_table = []
    latex_table.append(r"\begin{table}")
    latex_table.append(r"\centering")

    # Define columns depending on use_multirow
    if use_multirow:
        # Function + Method + columns for each dimension
        
        col_spec = "l l " + " ".join(["c"] * len(DIMS))
        header = ["Complexity", "Method"] + [f"$d = {d}$" for d in DIMS]
    else:
        # Method + columns for each dimension; function name per \multicolumn row
        col_spec = "l " + " ".join(["c"] * len(DIMS))
        header = ["Method"] + [f"Dim {d}" for d in DIMS]

    latex_table.append(r"\begin{tabular}{" + col_spec + "}")
    latex_table.append(r"\hline")
    latex_table.append(" & ".join(header) + r" \\")
    latex_table.append(r"\hline")

    # Helper to build the cell with median, upper, lower
    def build_quantile_string(func_id, dim_id, method_id,data_type_flag):
        # Last non-NaN from each array

        if not significantly_worse_at_end == None:
            significantly_worse = significantly_worse_at_end[func_id][dim_id][method_id] 
        else:
            significantly_worse = True

        val_median = get_last_non_nan_value(median_results_list[func_id][dim_id][method_id])
        if not data_type_flag == 'rank':  
            val_lower  = get_last_non_nan_value(lower_quantiles_list[func_id][dim_id][method_id])
            val_upper  = get_last_non_nan_value(upper_quantiles_list[func_id][dim_id][method_id])

        if val_median is None:
            # No valid median => no data for this cell
            return "-"

        if data_type_flag == 'cumulative': 
            # Format all as 2 decimals
            median_str = f"{val_median:.0f}"
            lower_str  = "-" if (val_lower is None) else f"{val_lower:.0f}"
            upper_str  = "-" if (val_upper is None) else f"{val_upper:.0f}"

        elif data_type_flag == 'simple': 
            # Format all as 2 decimals
            median_str = f"{val_median:.2f}"
            lower_str  = "-" if (val_lower is None) else f"{val_lower:.2f}"
            upper_str  = "-" if (val_upper is None) else f"{val_upper:.2f}"

        if data_type_flag == 'rank':  
            median_str = f"{val_median:.1f}"
            if not significantly_worse:
                cell_text = r"$\boldsymbol{" + median_str + r"}$"
            else:
                cell_text = r"$" + median_str + r"$"
        else:
            # Combine into median^(upper)_(lower)
            if not significantly_worse:
                raise NotImplementedError
            else:
                cell_text = f"${median_str}^{{{upper_str}}}_{{{lower_str}}}$"
        if data_type_flag == "rank":
            colored_text = cell_text
        else:
            # Now color the entire cell text based on the median
            colored_text = color_text_with_median(
                val_median, cell_text, global_min, global_max
            )
        return colored_text

    # Build the rows
    for func_idx, func_name in enumerate(function_names):
        if use_multirow:
            # Each function name in a multirow for all methods
            num_methods = len(METHODS)
            
            col_spec = "l l " + " ".join(["c"] * len(DIMS))
            header = ["Complexity", "Method"] + [f"$d = {d}$" for d in DIMS]
        



            for method_idx, method_name in enumerate(METHODS):
                row_cells = []
                for dim_idx in range(len(DIMS)):
                    row_cells.append(build_quantile_string(func_idx, dim_idx, method_idx,data_type_flag))

                if method_idx == 0:
                    # First row => use multirow for the function name
                    if expected_ls is None:
                        row_items = [
                            rf"\multirow{{{num_methods}}}{{*}}{{{func_name}}}",
                            method_name
                        ] + row_cells
                    else:
                        row_items = [
                            rf"\multirow{{{num_methods+1}}}{{*}}{{{func_name}}}",
                            method_name
                        ] + row_cells                        
                else:
                    # Subsequent rows => empty cell for function name
                    row_items = [" ", method_name] + row_cells


                latex_table.append(" & ".join(row_items) + r" \\")
            
            if not expected_ls is None:
                row_items = [" ",r"$\mathbb{E}[p(l)]$"]+[f"${expected_ls[func_idx,d]}$" for d in range(len(DIMS))]
                latex_table.append(" & ".join(row_items) + r" \\")    

            latex_table.append(r"\hline")

        else:
            # Put function name across the top via multicolumn (no bold)
            latex_table.append(
                rf"\multicolumn{{{len(DIMS) + 1}}}{{c}}{{{func_name}}} \\"
            )
            latex_table.append(r"\hline")

            for method_idx, method_name in enumerate(METHODS):
                row_entries = [method_name]
                for dim_idx in range(len(DIMS)):
                    row_entries.append(build_quantile_string(func_idx, dim_idx, method_idx))
                latex_table.append(" & ".join(row_entries) + r" \\")
            latex_table.append(r"\hline")

    latex_table.append(r"\end{tabular}")
    if data_type_flag == 'rank':
        latex_table.append(r"\caption{Your Caption Here}")
        latex_table.append(r"\label{tab:ranks_results_table}")
    else:
        latex_table.append(r"\caption{Your Caption Here}")
        latex_table.append(r"\label{tab:your_label}")
    latex_table.append(r"\end{table}")

    return "\n".join(latex_table)


if LESLOGEIONLY:
    prefix = "LES_EI_"
else:
    prefix = ""



# Generate the LaTeX table code with multirow=True
latex_code_multirow = create_latex_table(function_names, DIMS, METHOD_TABLE_NAMES, avrg_best_histories,[],[],
                                         data_type_flag = "rank",
                                         use_multirow=True,
                                         significantly_worse_at_end = significantly_worse_at_end_histories)#,
                                        
if WITHIN_MDL:
    with open(prefix+"ranks_results_table_within_mdl.tex", "w", encoding="utf-8") as f:
        f.write(latex_code_multirow)   
else:
    with open(prefix+"ranks_results_table.tex", "w", encoding="utf-8") as f:
        f.write(latex_code_multirow)




# Generate the LaTeX table code with multirow=True
latex_code_multirow = create_latex_table(function_names, DIMS, METHOD_TABLE_NAMES, cum_avrg_best_histories,[],[],
                                         data_type_flag = "rank",
                                         use_multirow=True,
                                         significantly_worse_at_end = cum_significantly_worse_at_end_histories) #,
                                         #expected_ls = excpected_ls)

if WITHIN_MDL:
    with open(prefix+"cum_ranks_results_table_within_mdl.tex", "w", encoding="utf-8") as f:
        f.write(latex_code_multirow)   
else:
    with open(prefix+"cum_ranks_results_table_table.tex", "w", encoding="utf-8") as f:
        f.write(latex_code_multirow)
        

